In [ ]:
import sys
import os
import numpy as np

gmat_path = os.environ["GMAT_PATH"]
assert gmat_path, "Please set the GMAT_PATH environment variable"

startup = os.path.join(gmat_path, "bin", "api_startup_file.txt")
assert os.path.exists(startup), "Cannot find " + startup

sys.path.insert(1, os.path.join(gmat_path, "bin"))
import gmatpy as gmat

gmat.Setup(startup)

In [ ]:
kwargs = {}

coe_mi = np.array(
    [
        6.54140000e03,
        6.00000000e-01,
        6.08802626e01,
        8.48999999e01,
        1.16785933e02,
        -1.15020532e-15,
    ]
)
earth_grav_path = os.path.join(gmat_path, "data", "gravity", "earth", "JGM3.cof")
moon_grav_path = os.path.join(gmat_path, "data", "gravity", "luna", "grgm900c.cof")

moon_ci = gmat.Construct("CoordinateSystem", "MOON_CI", "Luna", "BodyInertial")

# Spacecraft
sc = gmat.Construct("Spacecraft", "LunaOrbiter")
sc.SetField("DateFormat", "UTCGregorian")
sc.SetField("Epoch", "1 Jan 2020 12:00:00.000")
sc.SetField("CoordinateSystem", "MOON_CI")
# sc.SetField("DisplayStateType", "Keplerian")

# Orbital state
sc.SetField("SMA", coe_mi[0])
sc.SetField("ECC", coe_mi[1])
sc.SetField("INC", coe_mi[2])
sc.SetField("RAAN", coe_mi[3])
sc.SetField("AOP", coe_mi[4])
sc.SetField("TA", coe_mi[5])

# Spacecraft ballistic properties for the SRP and Drag models
if "SRPArea" in kwargs:
    sc.SetField("SRPArea", 2.5)
if "Cr" in kwargs:
    sc.SetField("Cr", 1.75)
if "DragArea" in kwargs:
    sc.SetField("DragArea", 1.8)
if "Cd" in kwargs:
    sc.SetField("Cd", 2.1)

sc.SetField("DryMass", 80)

# Force model
grav = gmat.Construct("GravityField")
grav.SetField("BodyName", "Luna")
grav.SetField("PotentialFile", moon_grav_path)
grav.SetField("Degree", 20)
grav.SetField("Order", 20)

fm = gmat.Construct("ForceModel", "FM")
fm.SetField("CentralBody", "Luna")
fm.SetField("PrimaryBodies", ["Luna"])
fm.SetField("PointMasses", ["Earth", "Sun"])
fm.AddForce(grav)

gmat.Initialize()

In [ ]:
Dt = 20 * 60.0
tf = 200 * 24 * 3600.0
N_steps = int(tf / Dt)
tfs = np.linspace(0, tf, N_steps)

gator = gmat.Construct("PrinceDormand45", "Gator")
pdprop = gmat.Construct("Propagator", "PDProp")
pdprop.SetReference(gator)
pdprop.SetReference(fm)
# pdprop.SetField("MinStep", Dt)
# pdprop.SetField("MaxStep", Dt)

pdprop.SetField("InitialStepSize", 60.0)
pdprop.SetField("Accuracy", 1.0e-8)
pdprop.SetField("MinStep", 0.0)

gmat.Initialize()

pdprop.AddPropObject(sc)
pdprop.PrepareInternals()
gator = pdprop.GetPropagator()

In [ ]:
import time
from tqdm import tqdm

rvs = np.zeros((N_steps, 6))
t_start = time.time()
for i in tqdm(range(N_steps)):
    gator.Step(Dt)
    rvs[i] = gator.GetState()
t_elapsed = time.time() - t_start
print(f"Elapsed time: {t_elapsed:.3f} s")
print(f"N_steps: {N_steps}")

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

n = 1
fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=rvs[::n, 0], y=rvs[::n, 1], z=rvs[::n, 2], mode="lines", name="Orbit"
    )
)
fig.update_layout(scene=dict(aspectmode="data"))
fig.show()